# S1 - Tái lập M0 (KE Recovery gốc) bằng engine QPT
Cổng: khớp báo cáo gốc (1.867 lệnh, +4.337,4 điểm, 890 lệnh thắng, vốn cuối 533,74 triệu) với thuế = 0, rồi chạy cấu hình chính có thuế 0,1%.
Cần trên Drive: `data/VN30F1M_5m_20171106-20261030.csv`, `results/S0/bars_5m.parquet`, `src/{backend,ke_research,strategies}`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/KE_Recovery'
import sys, os, json, hashlib
sys.path.insert(0, f'{ROOT}/src')
OUT = f'{ROOT}/results/S1'
os.makedirs(OUT, exist_ok=True)
import pandas as pd
from ke_research import runner
from strategies import ke_recovery as ke
# Mã KE gốc phải còn nguyên bản đã đóng băng
sha = hashlib.sha256(open(f'{ROOT}/src/strategies/ke_recovery.py','rb').read()).hexdigest()
assert sha == '487a835492e6c9b6b5a4aa8e468aedc397755b183970dfa18fda338e8ee478f3', 'ke_recovery.py đã bị sửa'
PARAMS = {k: v['default'] for k, v in ke.STRATEGY['params'].items()}
print(PARAMS)

In [ ]:
# P0: file 5m gốc, thuế = 0 -> phải trùng báo cáo cũ
old = pd.read_csv(f'{ROOT}/data/VN30F1M_5m_20171106-20261030.csv')
sig_old = ke.signals(old, PARAMS).to_numpy()
res_p0 = runner.run(old, sig_old, runner.make_config(tax_rate=0.0))
s_p0 = runner.summarize(res_p0)
print(json.dumps(s_p0, indent=1))
assert s_p0['trades'] == 1867 and s_p0['wins'] == 890 and abs(s_p0['net_points'] - 4337.4) < 0.05, 'P0 không tái lập được'
runner.trades_frame(res_p0).to_parquet(f'{OUT}/trades_M0_old5m_tax0.parquet', index=False)

In [ ]:
# Cấu hình chính: thuế 0,1% (tax_rate = 0.001), trên cả file 5m gốc (đến 30/10/2026)
res_tax_full = runner.run(old, sig_old, runner.make_config(tax_rate=0.001))
s_tax_full = runner.summarize(res_tax_full)
print(json.dumps(s_tax_full, indent=1))
runner.trades_frame(res_tax_full).to_parquet(f'{OUT}/trades_M0_old5m_tax0p1.parquet', index=False)

In [ ]:
# Dữ liệu chuẩn cho các bước sau: 5m gộp từ 1m (đến 17/07/2026), thuế 0,1%
b5 = pd.read_parquet(f'{ROOT}/results/S0/bars_5m.parquet')
sig_new = ke.signals(b5, PARAMS).to_numpy()
res_new = runner.run(b5, sig_new, runner.make_config(tax_rate=0.001))
s_new = runner.summarize(res_new)
print(json.dumps(s_new, indent=1))
runner.trades_frame(res_new).to_parquet(f'{OUT}/trades_M0_5mFrom1m_tax0p1.parquet', index=False)
json.dump({'p0_old5m_tax0': s_p0, 'old5m_tax0p1': s_tax_full, 'new5m_to_20260717_tax0p1': s_new, 'params': PARAMS},
          open(f'{OUT}/summary.json', 'w'), indent=1)